# Project Summary
In this project we analyze the effect of 2 different marketing strategies (the **treatments**) with the aim of maximizing the monetary return by presenting *the right people* with the right strategy.  
The basic idea is that in a group of customers there are 4 types of people: 
1. People who respond positively to being contacted with an offer. These should be the contact phase targets
2. People who buy anyway. Targeting them does not increase the returns.
3. People who would never buy. Targeting them does not increase the returns.
4. People who respond negatively. Targeting them will decrease the returns.   

With this analysis we aim at profiling the customers to figure out who is a type 1 customer, and therefore an optimal target for the campaign. 

# Phase 1: Data Cleaning  

**Raw data:** 
| Column | Values | Meaning | Role |
|---|---|---|---|
| `recency` | 1–12 | Months since last purchase | Pre-campaign feature |
| `history_segment` | `1) $0 - $100` … `7) $1,000 +` | Past-year spend, in bins | Feature (same information as `history`) |
| `history` | $ | Past-year spend | Feature |
| `mens` | 0/1 | Bought men's items in past year | Feature |
| `womens` | 0/1 | Bought women's items in past year | Feature (can be 1 together with `mens`) |
| `zip_code` | Urban / Surburban / Rural | Area type. "Surburban" is misspelled in the raw file | Feature |
| `newbie` | 0/1 | Became a customer in the past 12 months | Feature |
| `channel` | Phone / Web / Multichannel | Where they bought in the past year | Feature |
| `segment` | Mens E-Mail / Womens E-Mail / No E-Mail | Randomly assigned group | **Treatment** |
| `visit` | 0/1 | Visited the site within 2 weeks | Outcome |
| `conversion` | 0/1 | Bought within 2 weeks | Outcome |
| `spend` | $ | Amount spent within 2 weeks (0 for non-buyers, not missing) | Outcome |

In this notebook we organize the data, we clean it and we create a local SQL database 

In [ ]:
%load_ext autoreload
%autoreload 2

import duckdb

from digital_sales.get_data import build_database, download_raw
from digital_sales.paths import DB_PATH

In [ ]:
csv_path = download_raw()
build_database(csv_path)

In [11]:
con = duckdb.connect(str(DB_PATH), read_only=True)

con.sql("SHOW TABLES").show()

con.sql("""
    SELECT
        treatment,
        count(*)                         AS customers,
        round(avg(visit) * 100, 2)       AS visit_rate_pct,
        round(avg(conversion) * 100, 2)  AS conversion_rate_pct,
        round(avg(spend), 2)             AS spend_per_customer
    FROM analysis_base
    GROUP BY treatment
    ORDER BY treatment
""").show()

con.close() 

┌─────────────────────┐
│        name         │
│       varchar       │
├─────────────────────┤
│ analysis_base       │
│ campaign_assignment │
│ campaign_outcomes   │
│ customers           │
│ raw_hillstrom       │
└─────────────────────┘

┌──────────────┬───────────┬────────────────┬─────────────────────┬────────────────────┐
│  treatment   │ customers │ visit_rate_pct │ conversion_rate_pct │ spend_per_customer │
│   varchar    │   int64   │     double     │       double        │       double       │
├──────────────┼───────────┼────────────────┼─────────────────────┼────────────────────┤
│ mens_email   │     21307 │          18.28 │                1.25 │               1.42 │
│ no_email     │     21306 │          10.62 │                0.57 │               0.65 │
│ womens_email │     21387 │          15.14 │                0.88 │               1.08 │
└──────────────┴───────────┴────────────────┴─────────────────────┴────────────────────┘

